# 렌디(Rendy) · 가전 파손 판정 AI 실험 (코랩)

출고 사진과 반납 사진을 비교해 **새로 생긴 흠집·찍힘을 찾고, 길이를 cm로 재서 생활 마모 / 과실 파손 후보로 나누는** 파이프라인을 시험합니다.

| 단계 | 내용 |
|---|---|
| 1 | 데이터셋 만들기 — 가전 사진(합성 또는 내 사진)에 흠집·찍힘을 **정답 위치·실제 길이와 함께** 합성 |
| 2 | 이상 탐지 모델 **PatchCore** — 정상 사진만으로 학습 (ImageNet 사전학습 WideResNet50 특징 + 최근접 이웃) |
| 3 | 측정 — 이상 탐지 AUROC, 픽셀 AUROC, 흠집 길이 오차(cm), 생활 마모/파손 판정 정확도 |
| 4 | 데모 — 출고 사진 vs 반납 사진 → 사진 맞추기 → 새 흠집 표시 → 길이 → 판정 → 상태 점수 |

**실행 방법**: 메뉴 `런타임 > 런타임 유형 변경 > T4 GPU` → `런타임 > 모두 실행`. 처음 실행은 5~10분 정도 걸립니다.

> 주의: 기본 데이터는 **합성 데이터**라 실제 정확도가 아니라 "파이프라인이 동작하는지"를 보는 용도입니다. 실제 성능은 2-B에서 실제 가전 사진을 올려 다시 측정하세요.

## 0. 준비

In [ ]:
import os, json, math, random, zipfile, time
import numpy as np, cv2, torch, torch.nn.functional as F, torchvision
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from scipy.ndimage import gaussian_filter

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cpu':
    print('GPU가 꺼져 있어요. 런타임 > 런타임 유형 변경 > T4 GPU 를 켜면 훨씬 빨라요.')

# 한글 그래프 글꼴 (없으면 영어로 표시)
try:
    import matplotlib.font_manager as fm
    os.system('apt-get -qq install -y fonts-nanum > /dev/null 2>&1')
    path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    if os.path.exists(path):
        fm.fontManager.addfont(path); plt.rcParams['font.family'] = 'NanumGothic'
    plt.rcParams['axes.unicode_minus'] = False
except Exception as e:
    print('font skip', e)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1. 설정

In [ ]:
CFG = dict(
    image_size = 384,          # 모델 입력 크기 (px). 크게 하면 가는 흠집에 강하지만 느려져요
    device_width_cm = 30.0,    # 사진 속 가전 앞면의 실제 폭 (제습기 10L 약 30cm)
    wear_limit_cm = 2.0,       # 이 길이 이하면 생활 마모, 넘으면 과실 파손 후보 (기능명세서 부록 C-7)
    n_train_good = 60,         # 학습용 정상 사진 수
    n_val_good = 20,           # 기준값(임계값) 정하는 정상 사진 수
    n_test_good = 40,          # 시험용 정상 사진
    n_test_scratch = 80,       # 시험용 흠집 사진 (길이 0.5~5cm 고르게)
    n_test_dent = 30,          # 시험용 찍힘 사진
    coreset_ratio = 0.05,      # PatchCore 메모리 뱅크 비율
    use_my_photos = False,     # True면 2-B에서 내 가전 사진을 올려 사용
    out_dir = '/content/rendy_damage',
)
os.makedirs(CFG['out_dir'], exist_ok=True)
CFG

## 2. 데이터셋 만들기
### 2-A. 기본: 합성 가전(제습기 앞면) 사진
가전 앞면을 그려 "실물 사진 한 장"으로 쓰고, 찍을 때마다 각도·밝기·노이즈가 조금씩 다른 사진을 만듭니다.

In [ ]:
S = CFG['image_size']

def make_base_appliance(seed=0):
    # 합성 제습기 앞면 (흰 플라스틱 몸체 + 조작부 + 물통 창 + 통풍구)
    rng = np.random.default_rng(seed)
    img = np.zeros((S, S, 3), np.float32)
    # 배경: 벽 (밝은 베이지/회색 그라데이션)
    top = np.array([228, 226, 220], np.float32); bot = np.array([205, 203, 198], np.float32)
    for y in range(S):
        img[y] = top + (bot - top) * (y / S)
    img += rng.normal(0, 1.5, img.shape)
    # 몸체
    bw, bh = int(S * 0.62), int(S * 0.86)
    x0, y0 = (S - bw) // 2, (S - bh) // 2
    body = np.zeros((S, S), np.uint8)
    cv2.rectangle(body, (x0 + 20, y0), (x0 + bw - 20, y0 + bh), 255, -1)
    cv2.rectangle(body, (x0, y0 + 20), (x0 + bw, y0 + bh - 20), 255, -1)
    for cx, cy in [(x0 + 20, y0 + 20), (x0 + bw - 20, y0 + 20), (x0 + 20, y0 + bh - 20), (x0 + bw - 20, y0 + bh - 20)]:
        cv2.circle(body, (cx, cy), 20, 255, -1)
    shade = np.linspace(246, 232, S)[None, :, None] * np.ones((S, 1, 1))     # 왼쪽이 밝은 조명
    plastic = shade + rng.normal(0, 1.2, (S, S, 1))                          # 플라스틱 질감
    m = body.astype(bool)
    img[m] = np.repeat(plastic, 3, axis=2)[m]
    # 테두리 그림자
    edge = cv2.morphologyEx(body, cv2.MORPH_GRADIENT, np.ones((3, 3), np.uint8)).astype(bool)
    img[edge] = (170, 170, 168)
    # 조작부
    py0 = y0 + int(bh * 0.06)
    cv2.rectangle(img, (x0 + 30, py0), (x0 + bw - 30, py0 + 34), (60, 64, 72), -1)
    for i in range(4):
        cv2.circle(img, (x0 + 52 + i * 34, py0 + 17), 6, (120, 200, 180) if i == 0 else (150, 150, 150), -1)
    # 통풍구 줄
    for i in range(6):
        yy = py0 + 52 + i * 9
        cv2.line(img, (x0 + 40, yy), (x0 + bw - 40, yy), (200, 200, 198), 2)
    # 물통 창
    wy0 = y0 + int(bh * 0.55)
    cv2.rectangle(img, (x0 + 34, wy0), (x0 + bw - 34, y0 + bh - 30), (222, 226, 228), -1)
    cv2.rectangle(img, (x0 + 34, wy0), (x0 + bw - 34, y0 + bh - 30), (185, 188, 190), 2)
    cv2.putText(img, 'RENDY', (x0 + bw // 2 - 34, wy0 - 14), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (150, 155, 160), 2, cv2.LINE_AA)
    img = np.clip(img, 0, 255).astype(np.uint8)
    px_per_cm = bw / CFG['device_width_cm']
    # 흠집을 넣을 수 있는 곳 = 몸체에서 조작부·창 테두리 조금 피한 영역
    safe = cv2.erode(body, np.ones((15, 15), np.uint8))
    return img, safe, px_per_cm

BASES = [make_base_appliance(0)]
print('px/cm =', round(BASES[0][2], 2), '→ 2cm =', round(2 * BASES[0][2], 1), 'px')
plt.figure(figsize=(4, 4)); plt.imshow(BASES[0][0]); plt.axis('off'); plt.title('기본 가전 (합성)'); plt.show()

### 2-B. (선택) 내 가전 사진으로 만들기
`CFG['use_my_photos'] = True`로 바꾸고 이 칸을 실행하면 사진을 올릴 수 있어요.
- 같은 가전의 **앞면을 정면에서** 3~10장 찍어 올려 주세요 (흠집 없는 상태).
- 사진에 보이는 가로 폭(cm)을 `visible_width_cm`에 적어 주세요.

In [ ]:
visible_width_cm = 40.0   # 사진 가로 전체가 실제로 몇 cm인지 (가전 폭 + 여백)

if CFG['use_my_photos']:
    from google.colab import files
    up = files.upload()
    BASES = []
    for name, data in up.items():
        arr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
        if arr is None: continue
        arr = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
        h, w = arr.shape[:2]; c = min(h, w)
        arr = arr[(h - c) // 2:(h - c) // 2 + c, (w - c) // 2:(w - c) // 2 + c]
        arr = cv2.resize(arr, (S, S), interpolation=cv2.INTER_AREA)
        safe = np.zeros((S, S), np.uint8); safe[int(S * .15):int(S * .85), int(S * .2):int(S * .8)] = 255
        BASES.append((arr, safe, S / visible_width_cm))
    print('올린 사진', len(BASES), '장 · px/cm =', round(S / visible_width_cm, 2))
else:
    print('합성 가전을 사용해요 (use_my_photos=False)')

### 2-C. 흠집·찍힘 합성 + 촬영 조건 바꾸기

In [ ]:
def add_scratch(img, safe, px_per_cm, length_cm, rng):
    # 정답 길이(length_cm)의 흠집을 그리고 정답 마스크를 돌려줌
    L = length_cm * px_per_cm
    ys, xs = np.where(safe > 0)
    for _ in range(200):
        i = rng.integers(len(xs)); x1, y1 = xs[i], ys[i]
        ang = rng.uniform(0, np.pi)
        x2, y2 = x1 + L * np.cos(ang), y1 + L * np.sin(ang)
        if 0 <= x2 < S and 0 <= y2 < S and safe[int(y2), int(x2)] > 0:
            break
    # 살짝 휜 선 (2차 베지어)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    off = rng.normal(0, L * 0.06)
    cx, cy = mx - off * np.sin(ang), my + off * np.cos(ang)
    t = np.linspace(0, 1, max(8, int(L)))
    px = (1 - t) ** 2 * x1 + 2 * (1 - t) * t * cx + t ** 2 * x2
    py = (1 - t) ** 2 * y1 + 2 * (1 - t) * t * cy + t ** 2 * y2
    pts = np.stack([px, py], 1).astype(np.int32)
    mask = np.zeros((S, S), np.uint8)
    thick = int(rng.integers(2, 4))
    cv2.polylines(mask, [pts], False, 255, thick, cv2.LINE_AA)
    out = img.astype(np.float32)
    tone = rng.choice([255.0, 120.0])          # 하얗게 긁힘 또는 어둡게 긁힘
    a = (cv2.GaussianBlur(mask, (3, 3), 0) / 255.0 * rng.uniform(0.55, 0.85))[..., None]
    out = out * (1 - a) + tone * a
    return np.clip(out, 0, 255).astype(np.uint8), (mask > 60).astype(np.uint8)

def add_dent(img, safe, px_per_cm, diam_cm, rng):
    # 찍힘: 한쪽은 어둡고 한쪽은 밝은 오목한 자국
    r = diam_cm * px_per_cm / 2
    ys, xs = np.where(cv2.erode(safe, np.ones((int(2 * r) + 3, int(2 * r) + 3), np.uint8)) > 0)
    if len(xs) == 0: ys, xs = np.where(safe > 0)
    i = rng.integers(len(xs)); cx, cy = xs[i], ys[i]
    yy, xx = np.mgrid[0:S, 0:S].astype(np.float32)
    ang = rng.uniform(0, np.pi); ratio = rng.uniform(0.6, 1.0)
    dx, dy = xx - cx, yy - cy
    u = (dx * np.cos(ang) + dy * np.sin(ang)) / r
    v = (-dx * np.sin(ang) + dy * np.cos(ang)) / (r * ratio)
    d = np.sqrt(u ** 2 + v ** 2)
    inside = d < 1
    shade = np.where(inside, -np.cos(np.arctan2(v, u)) * (1 - d) * rng.uniform(35, 60), 0)
    out = np.clip(img.astype(np.float32) + shade[..., None], 0, 255).astype(np.uint8)
    return out, inside.astype(np.uint8)

def photo_variation(img, mask, rng):
    # 같은 물건을 다시 찍은 것처럼: 조금 돌리고·옮기고·밝기·노이즈·흐림·JPEG
    ang = rng.uniform(-3, 3); sc = rng.uniform(0.96, 1.04)
    tx, ty = rng.uniform(-8, 8, 2)
    M = cv2.getRotationMatrix2D((S / 2, S / 2), ang, sc); M[:, 2] += (tx, ty)
    im = cv2.warpAffine(img, M, (S, S), borderMode=cv2.BORDER_REFLECT)
    mk = cv2.warpAffine(mask, M, (S, S), flags=cv2.INTER_NEAREST)
    im = im.astype(np.float32) * rng.uniform(0.88, 1.12) + rng.uniform(-12, 12)
    im += rng.normal(0, rng.uniform(1.5, 4), im.shape)
    im = np.clip(im, 0, 255).astype(np.uint8)
    if rng.random() < 0.4: im = cv2.GaussianBlur(im, (3, 3), 0)
    ok, enc = cv2.imencode('.jpg', cv2.cvtColor(im, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, int(rng.integers(75, 95))])
    im = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    return im, mk, sc

def sample(kind, rng, length_cm=None):
    base, safe, ppc = BASES[rng.integers(len(BASES))]
    mask = np.zeros((S, S), np.uint8); L = 0.0
    img = base.copy()
    if kind == 'scratch':
        L = length_cm if length_cm else float(rng.uniform(0.5, 5.0))
        img, mask = add_scratch(img, safe, ppc, L, rng)
    elif kind == 'dent':
        L = length_cm if length_cm else float(rng.uniform(0.8, 4.0))
        img, mask = add_dent(img, safe, ppc, L, rng)
    img, mask, sc = photo_variation(img, mask, rng)
    return img, mask, L, ppc * sc

def build_dataset():
    rng = np.random.default_rng(SEED)
    root = CFG['out_dir']; rows = []
    plan = [('train', 'good', CFG['n_train_good']), ('val', 'good', CFG['n_val_good']),
            ('test', 'good', CFG['n_test_good']), ('test', 'scratch', CFG['n_test_scratch']), ('test', 'dent', CFG['n_test_dent'])]
    data = {'train': [], 'val': [], 'test': []}
    for split, kind, n in plan:
        os.makedirs(f'{root}/{split}/{kind}', exist_ok=True)
        if kind != 'good': os.makedirs(f'{root}/ground_truth/{kind}', exist_ok=True)
        for i in range(n):
            # 흠집 길이는 0.5~5cm를 고르게 → 2cm 기준 앞뒤가 골고루 섞이게
            Lc = 0.5 + 4.5 * (i + 0.5) / n if kind == 'scratch' else None
            img, mask, L, ppc = sample(kind if kind != 'good' else 'good', rng, Lc)
            name = f'{kind}_{i:03d}.png'
            cv2.imwrite(f'{root}/{split}/{kind}/{name}', cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            if kind != 'good': cv2.imwrite(f'{root}/ground_truth/{kind}/{name}', mask * 255)
            gt = '정상' if kind == 'good' else ('생활 마모' if L <= CFG['wear_limit_cm'] else '파손 후보')
            rows.append(dict(split=split, kind=kind, file=name, length_cm=round(L, 2), px_per_cm=round(ppc, 3), gt_class=gt))
            data[split].append(dict(img=img, mask=mask, kind=kind, length=L, ppc=ppc, gt=gt, file=name))
    import csv
    with open(f'{root}/labels.csv', 'w', newline='', encoding='utf-8-sig') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
    return data, rows

t0 = time.time()
DATA, ROWS = build_dataset()
print({k: len(v) for k, v in DATA.items()}, f'{time.time() - t0:.1f}s', '→', CFG['out_dir'])

fig, ax = plt.subplots(2, 5, figsize=(15, 6))
picks = [d for d in DATA['test'] if d['kind'] != 'good'][::22][:5]
for j, d in enumerate(picks):
    ax[0, j].imshow(d['img']); ax[0, j].set_title(f"{d['kind']} {d['length']:.1f}cm · {d['gt']}", fontsize=10)
    ax[1, j].imshow(d['mask'], cmap='gray'); ax[1, j].set_title('정답 마스크', fontsize=10)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

## 3. 모델: PatchCore (정상 사진만으로 학습하는 이상 탐지)
- ImageNet으로 사전학습된 WideResNet50의 중간 특징(layer2·layer3)을 작은 패치 단위로 뽑아 **정상 패치 메모리**를 만들어요.
- 새 사진의 각 패치가 메모리에서 가장 가까운 정상 패치와 얼마나 먼지 = 이상 점수. 멀수록 흠집일 가능성이 커요.
- 흠집 사진이 없어도 학습할 수 있다는 점이 렌디에 맞는 이유예요.

In [ ]:
class PatchCore:
    def __init__(self, coreset_ratio=0.05, feat_dim=384):
        w = torchvision.models.Wide_ResNet50_2_Weights.IMAGENET1K_V1
        net = torchvision.models.wide_resnet50_2(weights=w).eval().to(DEVICE)
        for p in net.parameters(): p.requires_grad_(False)
        self.net = net; self.ratio = coreset_ratio; self.feat_dim = feat_dim; self.bank = None
        self.mean = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

    @torch.no_grad()
    def features(self, imgs):
        x = torch.from_numpy(np.stack(imgs)).permute(0, 3, 1, 2).float().to(DEVICE) / 255.0
        x = (x - self.mean) / self.std
        n = self.net
        x = n.maxpool(n.relu(n.bn1(n.conv1(x))))
        f1 = n.layer2(n.layer1(x)); f2 = n.layer3(f1)
        f1 = F.avg_pool2d(f1, 3, 1, 1); f2 = F.avg_pool2d(f2, 3, 1, 1)
        f2 = F.interpolate(f2, size=f1.shape[-2:], mode='bilinear', align_corners=False)
        f = torch.cat([f1, f2], 1)                                   # B, 1536, h, w
        B, C, h, w = f.shape
        f = f.permute(0, 2, 3, 1).reshape(-1, C)
        f = F.adaptive_avg_pool1d(f.unsqueeze(1), self.feat_dim).squeeze(1)   # 차원 줄이기
        return f.reshape(B, h, w, self.feat_dim)

    @torch.no_grad()
    def fit(self, imgs, batch=8):
        feats = torch.cat([self.features(imgs[i:i + batch]).reshape(-1, self.feat_dim) for i in range(0, len(imgs), batch)])
        n = max(1, int(len(feats) * self.ratio))
        # 탐욕적 코어셋: 서로 멀리 떨어진 패치를 골라 메모리를 작게
        proj = feats @ torch.randn(self.feat_dim, 128, device=DEVICE) / math.sqrt(128)
        idx = [int(torch.randint(len(feats), (1,)))]
        dmin = torch.cdist(proj, proj[idx]).squeeze(1)
        for _ in range(n - 1):
            i = int(torch.argmax(dmin)); idx.append(i)
            dmin = torch.minimum(dmin, torch.cdist(proj, proj[i:i + 1]).squeeze(1))
        self.bank = feats[idx]
        print(f'패치 {len(feats):,}개 → 메모리 {len(self.bank):,}개')

    @torch.no_grad()
    def predict(self, imgs, batch=8):
        maps, scores = [], []
        for i in range(0, len(imgs), batch):
            f = self.features(imgs[i:i + batch]); B, h, w, D = f.shape
            d = torch.cat([torch.cdist(c, self.bank).min(1).values for c in f.reshape(-1, D).split(8192)])
            m = d.reshape(B, 1, h, w)
            m = F.interpolate(m, size=(S, S), mode='bilinear', align_corners=False).squeeze(1).cpu().numpy()
            m = np.stack([gaussian_filter(x, sigma=4) for x in m])
            maps.append(m); scores += [float(x.max()) for x in m]
        return np.concatenate(maps), np.array(scores)

t0 = time.time()
model = PatchCore(CFG['coreset_ratio'])
model.fit([d['img'] for d in DATA['train']])
print(f'학습 {time.time() - t0:.1f}s')

## 4. 측정

In [ ]:
val_maps, val_scores = model.predict([d['img'] for d in DATA['val']])
test = DATA['test']
t0 = time.time()
test_maps, test_scores = model.predict([d['img'] for d in test])
print(f'시험 {len(test)}장 추론 {time.time() - t0:.1f}s ({(time.time() - t0) / len(test) * 1000:.0f}ms/장)')

# 기준값: 정상 사진에서 나온 점수로 정함 (정상 사진은 거의 안 걸리게)
IMG_THR = float(np.percentile(val_scores, 99)) * 1.02
PIX_THR = float(np.percentile(val_maps, 99.9))
print('이미지 기준값', round(IMG_THR, 3), '| 픽셀 기준값', round(PIX_THR, 3))

def measure(amap, ppc):
    # 이상 지도 → 가장 큰 덩어리 → 긴 변 길이(cm)
    m = (amap > PIX_THR).astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m)
    if n <= 1: return 0.0, np.zeros_like(m)
    k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    comp = (lab == k).astype(np.uint8)
    cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    (_, _), (w, h), _ = cv2.minAreaRect(max(cnts, key=cv2.contourArea))
    return max(w, h) / ppc, comp

def verdict(detected, length_cm):
    if not detected: return '정상'
    return '생활 마모' if length_cm <= CFG['wear_limit_cm'] else '파손 후보'

y_true = np.array([0 if d['kind'] == 'good' else 1 for d in test])
img_auroc = roc_auc_score(y_true, test_scores)
pix_true, pix_score = [], []
prng = np.random.default_rng(0)
for d, m in zip(test, test_maps):
    pos = np.flatnonzero(d['mask'].ravel())                       # 흠집 픽셀은 전부
    neg = prng.choice(S * S, 3000, replace=False)                  # 나머지는 표본
    idx = np.concatenate([pos, neg])
    pix_true.append(d['mask'].ravel()[idx]); pix_score.append(m.ravel()[idx])
pix_auroc = roc_auc_score(np.concatenate(pix_true), np.concatenate(pix_score))
det = test_scores > IMG_THR
det_f1 = f1_score(y_true, det)

results = []
for d, m, s, dt in zip(test, test_maps, test_scores, det):
    L, comp = measure(m, d['ppc']) if dt else (0.0, None)
    if comp is not None and comp.sum() < 15: dt, L = False, 0.0          # 너무 작은 점은 무시
    results.append(dict(file=d['file'], kind=d['kind'], gt_len=d['length'], gt=d['gt'], score=s, detected=bool(dt), pred_len=L, pred=verdict(dt, L)))

dmg = [r for r in results if r['kind'] != 'good' and r['detected']]
len_mae = np.mean([abs(r['pred_len'] - r['gt_len']) for r in dmg]) if dmg else float('nan')
labels = ['정상', '생활 마모', '파손 후보']
acc = np.mean([r['pred'] == r['gt'] for r in results])
cm = confusion_matrix([r['gt'] for r in results], [r['pred'] for r in results], labels=labels)

print('\n===== 결과 (합성 데이터) =====')
print(f'이상 탐지 AUROC (사진 단위)    : {img_auroc:.3f}')
print(f'흠집 위치 AUROC (픽셀 단위)    : {pix_auroc:.3f}')
print(f'손상 사진 찾기 F1              : {det_f1:.3f}  (기준값 {IMG_THR:.3f})')
print(f'흠집 길이 오차 평균 (MAE)      : {len_mae:.2f} cm  (찾은 손상 {len(dmg)}장 기준)')
print(f'정상/생활 마모/파손 후보 정확도: {acc:.3f}')
for kind in ['good', 'scratch', 'dent']:
    rs = [r for r in results if r['kind'] == kind]
    print(f'  - {kind:8s} {len(rs):3d}장 · 찾은 비율 {np.mean([r["detected"] for r in rs]):.2f} · 판정 정확도 {np.mean([r["pred"] == r["gt"] for r in rs]):.2f}')

fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].hist(test_scores[y_true == 0], 30, alpha=.7, label='정상'); ax[0].hist(test_scores[y_true == 1], 30, alpha=.7, label='손상')
ax[0].axvline(IMG_THR, color='k', ls='--'); ax[0].legend(); ax[0].set_title('이상 점수 분포')
xs = [r['gt_len'] for r in dmg]; ys = [r['pred_len'] for r in dmg]
ax[1].scatter(xs, ys, s=14); mx = max(xs + ys + [5]); ax[1].plot([0, mx], [0, mx], 'k--')
ax[1].axvline(CFG['wear_limit_cm'], color='r', lw=.8); ax[1].axhline(CFG['wear_limit_cm'], color='r', lw=.8)
ax[1].set_xlabel('실제 길이 (cm)'); ax[1].set_ylabel('AI가 잰 길이 (cm)'); ax[1].set_title('길이 측정')
ax[2].imshow(cm, cmap='Blues'); ax[2].set_xticks(range(3)); ax[2].set_yticks(range(3))
ax[2].set_xticklabels(labels); ax[2].set_yticklabels(labels); ax[2].set_xlabel('AI 판정'); ax[2].set_ylabel('정답')
for i in range(3):
    for j in range(3): ax[2].text(j, i, cm[i, j], ha='center', va='center')
ax[2].set_title('판정 혼동 행렬')
plt.tight_layout(); plt.show()

### 예시 보기 (정답 vs AI가 찾은 곳)

In [ ]:
show = [i for i, r in enumerate(results) if r['kind'] != 'good'][::18][:6]
fig, ax = plt.subplots(3, len(show), figsize=(3.2 * len(show), 10))
for j, i in enumerate(show):
    d, r = test[i], results[i]
    ax[0, j].imshow(d['img']); ax[0, j].set_title(f"정답 {d['kind']} {d['length']:.1f}cm\n{d['gt']}", fontsize=9)
    ax[1, j].imshow(d['img']); ax[1, j].imshow(test_maps[i], cmap='jet', alpha=.45); ax[1, j].set_title('이상 지도', fontsize=9)
    ov = d['img'].copy(); ov[test_maps[i] > PIX_THR] = (255, 60, 60)
    ax[2, j].imshow(ov); ax[2, j].set_title(f"AI {r['pred_len']:.1f}cm · {r['pred']}", fontsize=9)
for a in ax.ravel(): a.axis('off')
plt.tight_layout(); plt.show()

## 5. 데모: 출고 사진 vs 반납 사진
1. 반납 사진을 출고 사진에 맞춰 겹쳐요 (특징점 ORB + 호모그래피).
2. 두 사진의 이상 지도를 빼서 **출고 때 없던 부분만** 남겨요 (받은 날부터 있던 흠은 고객 책임에서 제외).
3. 길이 → 판정 → 상태 점수(0~100).

In [ ]:
def align(src, dst):
    orb = cv2.ORB_create(3000)
    g1, g2 = cv2.cvtColor(src, cv2.COLOR_RGB2GRAY), cv2.cvtColor(dst, cv2.COLOR_RGB2GRAY)
    k1, d1 = orb.detectAndCompute(g1, None); k2, d2 = orb.detectAndCompute(g2, None)
    if d1 is None or d2 is None: return src
    mt = sorted(cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True).match(d1, d2), key=lambda m: m.distance)[:400]
    if len(mt) < 10: return src
    p1 = np.float32([k1[m.queryIdx].pt for m in mt]); p2 = np.float32([k2[m.trainIdx].pt for m in mt])
    H, _ = cv2.findHomography(p1, p2, cv2.RANSAC, 4.0)
    return cv2.warpPerspective(src, H, (S, S), borderMode=cv2.BORDER_REFLECT) if H is not None else src

def state_score(length_cm, kind_hint='scratch'):
    # 예시 규칙: 새 손상 1cm당 -6점, 2cm 넘으면 추가 -15점
    if length_cm <= 0: return 100
    return int(max(0, 100 - length_cm * 6 - (15 if length_cm > CFG['wear_limit_cm'] else 0)))

def demo(length_cm, kind='scratch', seed=7):
    rng = np.random.default_rng(seed)
    base, safe, ppc = BASES[0]
    ship, _, _ = photo_variation(base.copy(), np.zeros((S, S), np.uint8), rng)              # 출고 사진
    dmg = add_scratch(base.copy(), safe, ppc, length_cm, rng) if kind == 'scratch' else add_dent(base.copy(), safe, ppc, length_cm, rng)
    ret, gt_mask, sc = photo_variation(dmg[0], dmg[1], rng)                                 # 반납 사진 (다른 각도)
    ret_al = align(ret, ship)
    (m_ship, m_ret), _ = model.predict([ship, ret_al])
    new = np.where(m_ship > PIX_THR, 0, m_ret)                                             # 출고 때 없던 부분만
    L, comp = measure(new, ppc)
    found = comp is not None and comp.sum() >= 15
    L = L if found else 0.0
    v = verdict(found, L); sc_ = state_score(L)
    fee = '0원' if v != '파손 후보' else '케어 가입 0원 · 미가입 수리비 실비'
    fig, ax = plt.subplots(1, 4, figsize=(17, 4.6))
    ax[0].imshow(ship); ax[0].set_title('출고 사진')
    ax[1].imshow(ret); ax[1].set_title(f'반납 사진 (실제 {length_cm:.1f}cm {kind})')
    ax[2].imshow(ret_al); ax[2].imshow(m_ret - m_ship, cmap='jet', alpha=.45); ax[2].set_title('맞춘 뒤 달라진 정도')
    ov = ret_al.copy()
    if found: ov[comp > 0] = (255, 60, 60)
    ax[3].imshow(ov); ax[3].set_title(f'AI {L:.1f}cm → {v}\n상태 점수 {sc_} · 고객 부담 {fee}')
    for a in ax: a.axis('off')
    plt.tight_layout(); plt.show()

demo(1.5, 'scratch')
demo(3.5, 'scratch', seed=11)
demo(3.0, 'dent', seed=3)

## 6. 결과 저장 · 내려받기

In [ ]:
import csv
root = CFG['out_dir']
with open(f'{root}/results.csv', 'w', newline='', encoding='utf-8-sig') as f:
    w = csv.DictWriter(f, fieldnames=list(results[0].keys())); w.writeheader(); w.writerows(results)
summary = dict(image_auroc=round(img_auroc, 4), pixel_auroc=round(pix_auroc, 4), detect_f1=round(det_f1, 4),
               length_mae_cm=round(float(len_mae), 3), verdict_accuracy=round(float(acc), 4),
               img_threshold=IMG_THR, pix_threshold=PIX_THR, config=CFG, data='synthetic' if not CFG['use_my_photos'] else 'my_photos')
json.dump(summary, open(f'{root}/summary.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
zp = '/content/rendy_damage_dataset.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for dp, _, fs in os.walk(root):
        for fn in fs: z.write(os.path.join(dp, fn), os.path.relpath(os.path.join(dp, fn), '/content'))
print(json.dumps({k: v for k, v in summary.items() if k != 'config'}, ensure_ascii=False, indent=1))
try:
    from google.colab import files; files.download(zp)
except Exception:
    print('저장 위치:', zp)

## 결과 읽는 법 · 다음 단계
| 지표 | 뜻 | 발표에서 말하는 법 |
|---|---|---|
| 이상 탐지 AUROC | 손상 사진과 정상 사진을 얼마나 잘 구분하는지 (1.0이 완벽) | "손상 사진 구분 성능" |
| 픽셀 AUROC | 흠집 **위치**를 얼마나 정확히 짚는지 | "흠집 위치 정확도" |
| 길이 오차 MAE | AI가 잰 흠집 길이와 실제 길이 차이 (cm) | "평균 ±0.○cm 오차로 길이 측정" |
| 판정 정확도 | 정상 / 생활 마모 / 파손 후보를 맞힌 비율 | "운영자 판정 전 1차 분류 정확도" |

**꼭 같이 말할 것**: 기본 결과는 **합성 데이터**입니다. 실제 성능은
1. 실제 가전(제습기 등) 앞면을 같은 거리·각도로 30장 이상 찍어 `use_my_photos=True`로 다시 측정
2. 실제 흠집 사진을 모아 정답을 표시한 시험셋으로 측정 (공개 데이터로는 MVTec AD · VisA · 차량 손상 CarDD — 라이선스 확인)
3. 서비스 지표: 운영자 최종 판정과의 일치율, 이의 신청 비율

**개선 아이디어**: 촬영 가이드 프레임(같은 거리·각도), 흠집 라벨이 쌓이면 YOLO 분할 모델로 흠집 종류(긁힘·찍힘·깨짐) 분류, 가는 흠집용으로 `image_size` 키우기.

## 7. 실제 흠집 사진으로 확인하기
위 1~6은 **합성 흠집**이에요. 여기서는 실제 물건 사진으로 확인합니다. (위 칸들을 먼저 한 번 실행해 둔 상태에서 이 칸들만 실행해도 돼요)

**찍는 법**
1. 같은 물건을 **같은 거리·같은 각도·같은 밝기**에서 찍어요. 물건이 화면 가운데를 꽉 채우게.
2. **흠집 없는 부분/상태 사진 5장 이상** → 아래 첫 번째 업로드 (많을수록 좋아요, 10~20장 추천)
3. **흠집 난 사진 1장 이상** → 두 번째 업로드
   - 같은 물건의 흠집 부위를 같은 방식으로 찍으면 가장 정확해요.
   - 흠집 없는 물건이면 마스킹테이프·네임펜으로 가는 선을 그어 흠집을 흉내 내도 돼요.
4. 사진 가로 폭이 실제로 몇 cm인지(`REAL_WIDTH_CM`), 흠집 실제 길이를 알면 `REAL_LENGTHS`에 적어 주세요 (자로 재서).

In [ ]:
RUN_SECTION7 = False      # 같은 물건의 흠집 없는 사진 + 흠집 사진이 있을 때만 True로 바꾸고 실행
REAL_WIDTH_CM = 30.0          # 사진 가로 전체가 실제로 몇 cm인지
REAL_LENGTHS = {}             # 예: {'scratch1.jpg': 1.5, 'scratch2.jpg': 3.0}  (모르면 비워 두기)

from google.colab import files

def load_square(data):
    arr = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if arr is None: return None
    arr = cv2.cvtColor(arr, cv2.COLOR_BGR2RGB)
    h, w = arr.shape[:2]; c = min(h, w)
    arr = arr[(h - c) // 2:(h - c) // 2 + c, (w - c) // 2:(w - c) // 2 + c]
    return cv2.resize(arr, (S, S), interpolation=cv2.INTER_AREA)

REAL_GOOD, REAL_BAD = [], []
if RUN_SECTION7:
    print('① 흠집 없는 사진을 고르세요 (5장 이상)')
    up_good = files.upload()
    REAL_GOOD = [x for x in (load_square(v) for v in up_good.values()) if x is not None]
    print('② 흠집 난 사진을 고르세요 (1장 이상)')
    up_bad = files.upload()
    REAL_BAD = [(k, load_square(v)) for k, v in up_bad.items()]
    REAL_BAD = [(k, x) for k, x in REAL_BAD if x is not None]
    print('정상', len(REAL_GOOD), '장 · 흠집', len(REAL_BAD), '장')
else:
    print('7번은 건너뜀 (RUN_SECTION7 = False). 사진이 있으면 True로 바꾸고 이 칸부터 다시 실행하세요.')


In [ ]:
if not (REAL_GOOD and REAL_BAD):
    print('사진이 없어서 7번 결과는 건너뜀')
else:
    # 정상 사진이 적으면 각도·밝기를 조금씩 바꿔 늘려서 학습 (흠집 사진은 학습에 안 씀)
    rng_r = np.random.default_rng(1)
    REAL_PPC = S / REAL_WIDTH_CM
    train_r, val_r = [], []
    for i, im in enumerate(REAL_GOOD):
        (val_r if (i % 5 == 4 and len(REAL_GOOD) >= 5) else train_r).append(im)
    aug = []
    while len(train_r) + len(aug) < 40:
        im = train_r[len(aug) % len(train_r)]
        aug.append(photo_variation(im, np.zeros((S, S), np.uint8), rng_r)[0])
    train_r = train_r + aug
    if not val_r:   # 정상 사진이 5장 미만이면 기준값용도 변형으로 만듦
        val_r = [photo_variation(REAL_GOOD[i % len(REAL_GOOD)], np.zeros((S, S), np.uint8), rng_r)[0] for i in range(8)]

    real_model = PatchCore(CFG['coreset_ratio'])
    real_model.fit(train_r)
    vm, vs = real_model.predict(val_r)
    R_IMG_THR = float(vs.max()) * 1.02          # 정상 사진 최고 점수보다 높으면 손상
    R_PIX_THR = float(np.percentile(vm, 99.9))
    print('기준값 (사진)', round(R_IMG_THR, 3), '| (픽셀)', round(R_PIX_THR, 3))

    def measure_real(amap):
        m = (amap > R_PIX_THR).astype(np.uint8)
        n, lab, stats, _ = cv2.connectedComponentsWithStats(m)
        if n <= 1: return 0.0, np.zeros_like(m)
        k = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        comp = (lab == k).astype(np.uint8)
        cnts, _ = cv2.findContours(comp, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        (_, _), (w, h), _ = cv2.minAreaRect(max(cnts, key=cv2.contourArea))
        return max(w, h) / REAL_PPC, comp

    bm, bs = real_model.predict([x for _, x in REAL_BAD])
    gm, gs = real_model.predict(REAL_GOOD)
    rows_r = []
    fig, ax = plt.subplots(len(REAL_BAD), 3, figsize=(12, 4 * len(REAL_BAD)), squeeze=False)
    for i, ((name, im), amap, sc) in enumerate(zip(REAL_BAD, bm, bs)):
        found = sc > R_IMG_THR
        L, comp = measure_real(amap) if found else (0.0, None)
        v = verdict(found, L)
        true_L = REAL_LENGTHS.get(name)
        rows_r.append(dict(file=name, score=round(float(sc), 3), found=bool(found), ai_len_cm=round(L, 2), true_len_cm=true_L, verdict=v))
        ax[i, 0].imshow(im); ax[i, 0].set_title(f'{name}' + (f' (실제 {true_L}cm)' if true_L else ''), fontsize=10)
        ax[i, 1].imshow(im); ax[i, 1].imshow(amap, cmap='jet', alpha=.45); ax[i, 1].set_title(f'이상 점수 {sc:.2f} (기준 {R_IMG_THR:.2f})', fontsize=10)
        ov = im.copy()
        if comp is not None: ov[comp > 0] = (255, 60, 60)
        ax[i, 2].imshow(ov); ax[i, 2].set_title(f'AI: {"찾음" if found else "못 찾음"} · {L:.1f}cm → {v}', fontsize=10)
    for a in ax.ravel(): a.axis('off')
    plt.tight_layout(); plt.show()

    print('\n===== 실제 사진 결과 =====')
    print(f'흠집 사진 {len(REAL_BAD)}장 중 찾은 사진: {sum(r["found"] for r in rows_r)}장')
    print(f'흠집 없는 사진 {len(REAL_GOOD)}장 중 잘못 잡은 사진: {int((gs > R_IMG_THR).sum())}장  (학습에 쓴 사진이라 참고만)')
    withL = [r for r in rows_r if r['true_len_cm'] and r['found']]
    if withL:
        print('길이 오차 평균:', round(np.mean([abs(r['ai_len_cm'] - r['true_len_cm']) for r in withL]), 2), 'cm')
    for r in rows_r: print(r)
